# Portfolio Risk Engine Example

This notebook is the living example for the project. As new features are added, we can extend this notebook with new sections instead of creating disconnected demos.


In [31]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'bigdata').is_dir() and (candidate / 'README.md').exists():
            return candidate
    raise RuntimeError('Could not locate the project root directory.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root added to sys.path: {PROJECT_ROOT}')

Project root added to sys.path: /Users/jiangzhenhao/Desktop/bigdata


## 1. Universe Definition

The current version uses a multi-asset ETF universe across equities, fixed income, commodities, and an FX proxy.

In [32]:
from bigdata.universe import ASSET_LIST, ASSET_CLASS_MAPPING, ASSET_UNIVERSE

print('Asset list:')
print(ASSET_LIST)

print('\nAsset class mapping:')
print(ASSET_CLASS_MAPPING)

ASSET_UNIVERSE

Asset list:
['SPY', 'QQQ', 'IWM', 'TLT', 'IEF', 'GLD', 'USO', 'UUP', 'EEM']

Asset class mapping:
{'SPY': 'Equities', 'QQQ': 'Equities', 'IWM': 'Equities', 'TLT': 'Fixed Income', 'IEF': 'Fixed Income', 'GLD': 'Commodities', 'USO': 'Commodities', 'UUP': 'FX Proxy', 'EEM': 'Equities'}


[{'ticker': 'SPY', 'asset_class': 'Equities'},
 {'ticker': 'QQQ', 'asset_class': 'Equities'},
 {'ticker': 'IWM', 'asset_class': 'Equities'},
 {'ticker': 'TLT', 'asset_class': 'Fixed Income'},
 {'ticker': 'IEF', 'asset_class': 'Fixed Income'},
 {'ticker': 'GLD', 'asset_class': 'Commodities'},
 {'ticker': 'USO', 'asset_class': 'Commodities'},
 {'ticker': 'UUP', 'asset_class': 'FX Proxy'},
 {'ticker': 'EEM', 'asset_class': 'Equities'}]

## 2. Data Pipeline Example

This section runs the end-to-end data pipeline:

- download daily adjusted close prices
- align to a common business-day calendar
- forward fill internal gaps
- compute daily simple returns
- create rolling train/test windows


In [33]:
from bigdata.data_pipeline import build_data_pipeline

pipeline_result = build_data_pipeline(
    tickers=ASSET_LIST,
    start_date='2020-01-01',
    missing_ratio_threshold=0.10,
    winsorize_limit=0.01,
    train_window=252,
    test_window=21,
    step_size=21,
)

pipeline_result

DataPipelineResult(clean_return_matrix=Ticker           EEM       GLD       IEF       IWM       QQQ       SPY  \
2020-01-03 -0.018567  0.013269  0.006683 -0.003920 -0.009160 -0.007572   
2020-01-06 -0.002448  0.010490 -0.001076  0.001332  0.006443  0.003815   
2020-01-07 -0.000669  0.003935 -0.001437 -0.003326 -0.000139 -0.002812   
2020-01-08  0.005805 -0.007502 -0.002338  0.003095  0.007516  0.005329   
2020-01-09  0.006659 -0.005652  0.000721  0.001210  0.008473  0.006781   
...              ...       ...       ...       ...       ...       ...   
2026-03-26 -0.033960 -0.031455 -0.008075 -0.017393 -0.023868 -0.017859   
2026-03-27 -0.004868  0.030027  0.000106 -0.017540 -0.019537 -0.017052   
2026-03-30 -0.008152 -0.000289  0.007082 -0.014356 -0.007643 -0.003343   
2026-03-31  0.033420  0.030027  0.001784  0.035015  0.033854  0.029068   
2026-04-01  0.007748  0.017500 -0.004191  0.006290  0.012353  0.007534   

Ticker           TLT       USO       UUP  
2020-01-03  0.015400  0.02888

In [34]:
pipeline_result.clean_price_matrix.tail()

Ticker,EEM,GLD,IEF,IWM,QQQ,SPY,TLT,USO,UUP
2026-03-26,55.470001,400.640015,94.589996,247.440002,573.789978,645.090027,85.767311,117.260002,27.809999
2026-03-27,55.200001,414.700012,94.599998,243.100006,562.580017,634.090027,85.299179,124.199997,27.840000
2026-03-30,54.750000,414.579987,95.269997,239.610001,558.280029,631.969971,86.434639,129.830002,27.980000
2026-03-31,56.790001,430.290009,95.440002,248.000000,577.179993,650.340027,86.345001,127.250000,27.780001
2026-04-01,57.230000,437.820007,95.040001,249.559998,584.309998,655.239990,86.260002,124.089996,27.730000


In [35]:
pipeline_result.clean_return_matrix.tail()

Ticker,EEM,GLD,IEF,IWM,QQQ,SPY,TLT,USO,UUP
2026-03-26,-0.033960,-0.031455,-0.008075,-0.017393,-0.023868,-0.017859,-0.008406,0.034130,0.003971
2026-03-27,-0.004868,0.030027,0.000106,-0.017540,-0.019537,-0.017052,-0.005458,0.059185,0.001079
2026-03-30,-0.008152,-0.000289,0.007082,-0.014356,-0.007643,-0.003343,0.013311,0.045330,0.005029
2026-03-31,0.033420,0.030027,0.001784,0.035015,0.033854,0.029068,-0.001037,-0.019872,-0.007148
2026-04-01,0.007748,0.017500,-0.004191,0.006290,0.012353,0.007534,-0.000984,-0.024833,-0.001800


In [36]:
pipeline_result.missing_ratio_by_asset

Ticker
EEM    0.03681
GLD    0.03681
IEF    0.03681
IWM    0.03681
QQQ    0.03681
SPY    0.03681
TLT    0.03681
USO    0.03681
UUP    0.03681
dtype: float64

In [37]:
pipeline_result.dropped_assets

[]

In [38]:
first_window = pipeline_result.training_test_windows[0]

print('Train window:', first_window.train_start, '->', first_window.train_end)
print('Test window:', first_window.test_start, '->', first_window.test_end)
print('\nTrain shape:', first_window.train_data.shape)
print('Test shape:', first_window.test_data.shape)

Train window: 2020-01-03 00:00:00 -> 2020-12-21 00:00:00
Test window: 2020-12-22 00:00:00 -> 2021-01-19 00:00:00

Train shape: (252, 9)
Test shape: (21, 9)


## 3. Portfolio Construction Example

The next layer uses rolling train/test windows to compute stable portfolio weights without look-ahead bias.

In [39]:
from bigdata.portfolio_construction import run_rolling_portfolio_construction

portfolio_result = run_rolling_portfolio_construction(
    windows=pipeline_result.training_test_windows,
    strategy='min_variance',
    shrinkage_intensity=0.10,
    shrinkage_target='diagonal',
    max_weight=0.30,
)

portfolio_result

PortfolioConstructionResult(rolling_weights=Ticker               EEM       GLD  IEF           IWM           QQQ       SPY  \
rebalance_date                                                                  
2020-12-22      0.063953  0.095269  0.3  2.214399e-02  0.000000e+00  0.064496   
2021-01-20      0.060373  0.093548  0.3  2.833651e-02  1.418074e-18  0.060161   
2021-02-18      0.045452  0.098377  0.3  2.683528e-02  0.000000e+00  0.073678   
2021-03-19      0.044215  0.105952  0.3  3.012854e-02  0.000000e+00  0.067010   
2021-04-19      0.031571  0.111943  0.3  2.046394e-02  9.947280e-19  0.090985   
...                  ...       ...  ...           ...           ...       ...   
2025-10-21      0.075091  0.121764  0.3  0.000000e+00  0.000000e+00  0.092343   
2025-11-19      0.093841  0.107220  0.3  0.000000e+00  0.000000e+00  0.057200   
2025-12-18      0.087361  0.113941  0.3  0.000000e+00  7.325294e-18  0.063328   
2026-01-16      0.084342  0.104554  0.3  2.434440e-17  0.000000e+

In [40]:
portfolio_result.rolling_weights.tail()

Ticker,EEM,GLD,IEF,IWM,QQQ,SPY,TLT,USO,UUP
rebalance_date,,,,,,,,,
2025-10-21,0.075091,0.121764,0.3,0.000000e+00,0.000000e+00,0.092343,0.095577,0.015225,0.3
2025-11-19,0.093841,0.107220,0.3,0.000000e+00,0.000000e+00,0.057200,0.118711,0.023028,0.3
2025-12-18,0.087361,0.113941,0.3,0.000000e+00,7.325294e-18,0.063328,0.117289,0.018081,0.3
2026-01-16,0.084342,0.104554,0.3,2.434440e-17,0.000000e+00,0.063209,0.127574,0.020321,0.3
2026-02-16,0.079079,0.092481,0.3,0.000000e+00,1.788790e-17,0.062169,0.147092,0.019179,0.3


In [41]:
portfolio_result.portfolio_returns.tail()

2026-03-10   -0.001101
2026-03-11   -0.001968
2026-03-12   -0.003945
2026-03-13   -0.000278
2026-03-16    0.002740
Freq: B, Name: portfolio_return, dtype: float64

In [42]:
portfolio_result.turnover_series.tail()

rebalance_date
2025-10-21    0.038653
2025-11-19    0.099375
2025-12-18    0.025698
2026-01-16    0.025049
2026-02-16    0.039037
Name: turnover, dtype: float64

## 4. Risk Engine Example

This section computes rolling VaR and ES from the portfolio return series using historical, parametric, and Monte Carlo methods.

In [43]:
from bigdata.risk_engine import run_rolling_risk_engine

risk_result = run_rolling_risk_engine(
    portfolio_returns=portfolio_result.portfolio_returns,
    window_size=252,
    confidence_level=0.95,
    n_simulations=10000,
    random_state=42,
)

risk_result

RiskEngineResult(var_series=            historical  parametric  monte_carlo
date                                           
2021-12-09    0.004060    0.003865     0.003905
2021-12-10    0.004060    0.003871     0.003915
2021-12-13    0.004060    0.003860     0.003909
2021-12-14    0.004060    0.003860     0.003822
2021-12-15    0.004060    0.003880     0.003890
...                ...         ...          ...
2026-03-10    0.003223    0.004022     0.004118
2026-03-11    0.003223    0.004026     0.003982
2026-03-12    0.003223    0.004045     0.003920
2026-03-13    0.003352    0.004086     0.004040
2026-03-16    0.003352    0.004076     0.004040

[1113 rows x 3 columns], es_series=            historical  parametric  monte_carlo
date                                           
2021-12-09    0.005460    0.004885     0.004947
2021-12-10    0.005460    0.004890     0.004898
2021-12-13    0.005460    0.004880     0.004947
2021-12-14    0.005460    0.004880     0.004792
2021-12-15    0.005460  

In [44]:
risk_result.var_series.tail()

,historical,parametric,monte_carlo
date,,,
2026-03-10,0.003223,0.004022,0.004118
2026-03-11,0.003223,0.004026,0.003982
2026-03-12,0.003223,0.004045,0.003920
2026-03-13,0.003352,0.004086,0.004040
2026-03-16,0.003352,0.004076,0.004040


In [45]:
risk_result.es_series.tail()

,historical,parametric,monte_carlo
date,,,
2026-03-10,0.006552,0.005182,0.005281
2026-03-11,0.006552,0.005186,0.005102
2026-03-12,0.006552,0.005207,0.005045
2026-03-13,0.006607,0.005255,0.005197
2026-03-16,0.006607,0.005243,0.005270


In [46]:
risk_result.pnl_distribution.head()

,historical,parametric,monte_carlo
0,0.005237,-0.001331,0.002629
1,0.003568,0.002889,-0.006687
2,0.002951,0.001794,-0.000051
3,0.001136,0.003508,-0.000740
4,0.003502,0.002988,0.004460


## 5. Stress Testing Example

This section replays historical crisis windows and identifies the worst realized periods using the current portfolio weights.

In [47]:
from bigdata.stress_testing import find_worst_historical_periods, run_historical_stress_scenarios

current_weights = portfolio_result.rolling_weights.iloc[-1]

historical_stress = run_historical_stress_scenarios(
    asset_returns=pipeline_result.clean_return_matrix,
    current_weights=current_weights,
)

worst_periods = find_worst_historical_periods(
    asset_returns=pipeline_result.clean_return_matrix,
    current_weights=current_weights,
)

historical_stress.scenario_loss_table

,Start Date,End Date,Scenario Return,Loss
Scenario,,,,
COVID,2020-02-20,2020-03-23,-0.001816,0.001816


In [48]:
historical_stress.stress_contribution_table

,EEM,GLD,IEF,IWM,QQQ,SPY,TLT,USO,UUP
Scenario,,,,,,,,,
COVID,0.013553,0.003438,-0.017812,0.0,4.193900e-18,0.015705,-0.019009,0.007868,-0.001928


In [49]:
worst_periods.scenario_loss_table

,Start Date,End Date,Scenario Return,Loss
Scenario,,,,
Worst Day,2020-03-11,2020-03-11,-0.012401,0.012401
Worst Week,2025-04-02,2025-04-08,-0.027792,0.027792
Worst Month,2022-08-30,2022-09-27,-0.042143,0.042143


## 6. Macro Factor Risk Decomposition Example

This section uses selected ETF returns as macro factor proxies and decomposes portfolio risk into systematic and idiosyncratic components.

In [50]:
from bigdata.macro_factor import run_macro_factor_risk_decomposition

macro_result = run_macro_factor_risk_decomposition(
    portfolio_returns=portfolio_result.portfolio_returns,
    asset_returns=pipeline_result.clean_return_matrix,
    factor_set=['SPY', 'TLT', 'GLD', 'UUP'],
    window_size=252,
)

macro_result

MacroFactorRiskResult(rolling_beta=                 SPY       TLT       GLD       UUP
date                                              
2021-12-09  0.171233  0.186333  0.128855  0.246965
2021-12-10  0.171259  0.185936  0.129075  0.245469
2021-12-13  0.171673  0.185909  0.129176  0.245633
2021-12-14  0.172161  0.185351  0.129062  0.244900
2021-12-15  0.172246  0.185368  0.129169  0.244938
...              ...       ...       ...       ...
2026-03-10  0.157390  0.229713  0.137056  0.270445
2026-03-11  0.157365  0.230042  0.137202  0.270049
2026-03-12  0.156719  0.230024  0.137288  0.270636
2026-03-13  0.156481  0.230038  0.137196  0.270858
2026-03-16  0.156342  0.229954  0.137051  0.272455

[1113 rows x 4 columns], factor_covariance_by_date={Timestamp('2021-12-09 00:00:00'):           SPY           TLT       GLD           UUP
SPY  0.000062 -7.907679e-06  0.000011 -7.062042e-06
TLT -0.000008  7.154390e-05  0.000016 -7.866194e-07
GLD  0.000011  1.602381e-05  0.000072 -1.461992e-05
UUP -0.

In [51]:
macro_result.rolling_beta.tail()

,SPY,TLT,GLD,UUP
date,,,,
2026-03-10,0.157390,0.229713,0.137056,0.270445
2026-03-11,0.157365,0.230042,0.137202,0.270049
2026-03-12,0.156719,0.230024,0.137288,0.270636
2026-03-13,0.156481,0.230038,0.137196,0.270858
2026-03-16,0.156342,0.229954,0.137051,0.272455


In [52]:
macro_result.systematic_risk_series.tail()

2026-03-10    0.000007
2026-03-11    0.000007
2026-03-12    0.000007
2026-03-13    0.000007
2026-03-16    0.000007
Name: systematic_risk, dtype: float64

In [53]:
macro_result.idiosyncratic_risk_series.tail()

2026-03-10    4.947707e-07
2026-03-11    4.911052e-07
2026-03-12    4.912260e-07
2026-03-13    4.914366e-07
2026-03-16    4.981466e-07
Name: idiosyncratic_risk, dtype: float64

In [54]:
macro_result.factor_risk_contribution_by_date.tail()

,SPY,TLT,GLD,UUP
date,,,,
2026-03-10,0.000002,0.000002,0.000002,1.800929e-07
2026-03-11,0.000002,0.000002,0.000002,1.840000e-07
2026-03-12,0.000002,0.000003,0.000002,1.739690e-07
2026-03-13,0.000002,0.000003,0.000002,1.520674e-07
2026-03-16,0.000002,0.000003,0.000002,1.572932e-07


## 7. Integrated Risk Reporting Example

This section combines portfolio, risk, stress, and macro-factor outputs into a single daily report with limit monitoring.

In [55]:
from bigdata.reporting import build_integrated_risk_report

report = build_integrated_risk_report(
    portfolio_result=portfolio_result,
    risk_result=risk_result,
    stress_result=worst_periods,
    macro_result=macro_result,
)

report.summary_table

,portfolio_return,volatility,VaR_hist,VaR_param,VaR_mc,ES_hist,ES_param,ES_mc,stress_loss,stress_scenario,...,rates_beta,commodity_beta,fx_beta,systematic_share,idiosyncratic_share,top_position,top_position_weight,top_factor,top_factor_contribution,turnover
date,,,,,,,,,,,,,,,,,,,,,
2026-03-16,0.00274,0.002793,0.003352,0.004076,0.00404,0.006607,0.005243,0.00527,0.042143,Worst Month,...,0.229954,0.137051,0.272455,0.936517,0.063483,IEF,0.3,TLT,0.000003,0.039037


In [56]:
report.factor_summary_table

,beta,risk_contribution
SPY,0.156342,2.336340e-06
TLT,0.229954,2.506889e-06
GLD,0.137051,2.348199e-06
UUP,0.272455,1.572932e-07


In [57]:
report.top_positions_table

,weight,abs_weight
Ticker,,
IEF,0.300000,0.300000
UUP,0.300000,0.300000
TLT,0.147092,0.147092


In [58]:
report.limit_check_table

,date,value,limit,status
metric_name,,,,
var_hist,2026-03-16,0.003352,0.03,OK
es_hist,2026-03-16,0.006607,0.04,OK
beta_abs,2026-03-16,0.272455,0.80,OK
turnover,2026-03-16,0.039037,0.25,OK
equity_beta,2026-03-16,0.156342,0.60,OK
stress_loss,2026-03-16,0.042143,0.12,OK


In [59]:
report.breach_log

""


In [60]:
report.alert_summary

date                        2026-03-16 00:00:00
message    All monitored metrics within limits.
dtype: object

## 8. Daily Risk Dashboard Example

This section builds a latest-date dashboard from the reporting outputs only, without performing new risk calculations.

In [61]:
from bigdata.dashboard import build_daily_risk_dashboard, render_dashboard_html

dashboard = build_daily_risk_dashboard(
    risk_summary_table=report.summary_table,
    factor_summary_table=report.factor_summary_table,
    top_positions_table=report.top_positions_table,
    limit_check_table=report.limit_check_table,
    alert_summary=report.alert_summary,
)

dashboard.headline

{'date': '2026-03-16',
 'portfolio_return': '0.27%',
 'VaR_hist': '0.34%',
 'ES_hist': '0.66%',
 'risk_status': 'OK'}

In [62]:
dashboard.risk_metrics

{'volatility': '0.28%',
 'VaR_hist': '0.34%',
 'VaR_param': '0.41%',
 'VaR_mc': '0.40%',
 'ES_hist': '0.66%',
 'ES_param': '0.52%',
 'ES_mc': '0.53%'}

In [63]:
dashboard.stress

{'worst_day_loss': '1.24%',
 'worst_week_loss': '2.78%',
 'worst_month_loss': '4.21%'}

In [64]:
dashboard.factor_exposure

{'equity_exposure': '15.63%',
 'rates_exposure': '23.00%',
 'commodity_exposure': '13.71%',
 'fx_exposure': '27.25%'}

In [65]:
dashboard.top_positions

,weight,abs_weight
Ticker,,
IEF,30.00%,30.00%
UUP,30.00%,30.00%
TLT,14.71%,14.71%


In [66]:
dashboard.limit_status['table']

,date,value,limit,status
metric_name,,,,
var_hist,2026-03-16,0.003352,0.03,OK
es_hist,2026-03-16,0.006607,0.04,OK
beta_abs,2026-03-16,0.272455,0.80,OK
turnover,2026-03-16,0.039037,0.25,OK
equity_beta,2026-03-16,0.156342,0.60,OK
stress_loss,2026-03-16,0.042143,0.12,OK


In [67]:
html_dashboard = render_dashboard_html(dashboard)
html_dashboard[:1000]

'<html><body><section><h2>Headline</h2><table><tr><th>date</th><td>2026-03-16</td></tr><tr><th>portfolio_return</th><td>0.27%</td></tr><tr><th>VaR_hist</th><td>0.34%</td></tr><tr><th>ES_hist</th><td>0.66%</td></tr><tr><th>risk_status</th><td>OK</td></tr></table></section><section><h2>Risk Metrics</h2><table><tr><th>volatility</th><td>0.28%</td></tr><tr><th>VaR_hist</th><td>0.34%</td></tr><tr><th>VaR_param</th><td>0.41%</td></tr><tr><th>VaR_mc</th><td>0.40%</td></tr><tr><th>ES_hist</th><td>0.66%</td></tr><tr><th>ES_param</th><td>0.52%</td></tr><tr><th>ES_mc</th><td>0.53%</td></tr></table></section><section><h2>Stress</h2><table><tr><th>worst_day_loss</th><td>1.24%</td></tr><tr><th>worst_week_loss</th><td>2.78%</td></tr><tr><th>worst_month_loss</th><td>4.21%</td></tr></table></section><section><h2>Factor Exposure</h2><table><tr><th>equity_exposure</th><td>15.63%</td></tr><tr><th>rates_exposure</th><td>23.00%</td></tr><tr><th>commodity_exposure</th><td>13.71%</td></tr><tr><th>fx_exposure<